# Vision Classification Training RunPod Pipeline

RunPod에서 상위 사고유형 4개 라벨 기준으로 학습 데이터를 다운로드하고, 프레임을 추출한 뒤 ResNet18 baseline 학습까지 한 번에 실행하는 노트북입니다.

대상 라벨:
- 차대차
- 차대보행자
- 차대이륜차
- 차대자전거

기본 설정은 라벨별 700개, 총 2,800개 영상입니다.

## 1. Project path and run configuration

RunPod에서는 보통 `/workspace/SKN27-FINAL-3Team`에서 실행합니다. 로컬에서 열어도 현재 위치를 기준으로 프로젝트 루트를 찾습니다.

In [ ]:
from pathlib import Path
import csv
import json
import shutil
import subprocess
import sys
from collections import Counter
from datetime import datetime

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "scripts":
    PROJECT_ROOT = PROJECT_ROOT.parent
if (Path("/workspace/SKN27-FINAL-3Team")).exists():
    PROJECT_ROOT = Path("/workspace/SKN27-FINAL-3Team")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("python:", sys.executable)

MANIFEST_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/manifests"
RAW_VIDEO_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/raw_videos"
FRAME_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/frames"
MODEL_DIR = PROJECT_ROOT / "storage/vision/models/classification"

SAMPLE_MANIFEST = MANIFEST_DIR / "sample_700_coarse_manifest.csv"
DOWNLOAD_MANIFEST = MANIFEST_DIR / "train_700_download_manifest.csv"
FRAME_MANIFEST = MANIFEST_DIR / "frame_manifest_train_700_f8.csv"

RUN_INSTALL_REQUIREMENTS = True
RUN_DOWNLOAD = True
RUN_EXTRACT_FRAMES = True
RUN_TRAIN = True

PER_LABEL = 700
FRAMES_PER_VIDEO = 8
EPOCHS = 5
BATCH_SIZE = 32
LEARNING_RATE = 0.001
IMAGE_SIZE = 224
SEED = 42
DETERMINISTIC = True
FREEZE_BACKBONE = True
PRETRAINED = True
DEVICE = "auto"
NUM_WORKERS = 4

# Set RUN_EXPERIMENTS=True to run several training configurations in sequence.
RUN_EXPERIMENTS = False
EXPERIMENTS = [
    {
        "name": "exp1_freeze_lr1e-3_e5",
        "epochs": 5,
        "batch_size": 32,
        "learning_rate": 0.001,
        "freeze_backbone": True,
    },
    {
        "name": "exp2_unfreeze_lr1e-4_e10",
        "epochs": 10,
        "batch_size": 32,
        "learning_rate": 0.0001,
        "freeze_backbone": False,
    },
    {
        "name": "exp3_unfreeze_lr3e-5_e10",
        "epochs": 10,
        "batch_size": 32,
        "learning_rate": 0.00003,
        "freeze_backbone": False,
    },
]

print("SAMPLE_MANIFEST:", SAMPLE_MANIFEST)
print("DOWNLOAD_MANIFEST:", DOWNLOAD_MANIFEST)
print("FRAME_MANIFEST:", FRAME_MANIFEST)

## 2. Environment and disk check

전체 2,800개 영상을 받으므로 RunPod 디스크 여유 공간을 먼저 확인합니다.

In [ ]:
usage = shutil.disk_usage(PROJECT_ROOT)
print("disk_total_gb:", round(usage.total / 1024**3, 2))
print("disk_used_gb:", round(usage.used / 1024**3, 2))
print("disk_free_gb:", round(usage.free / 1024**3, 2))

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda_available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("torch_check_failed:", exc)

## 3. Install requirements

RunPod 새 Pod에서는 먼저 requirements를 설치합니다. 이미 설치되어 있으면 대부분 `Requirement already satisfied`로 지나갑니다.

In [ ]:
def run_command(command, *, enabled=True, timeout=None):
    print()
    print("$", " ".join(map(str, command)))
    if not enabled:
        print("SKIPPED")
        return None

    process = subprocess.Popen(
        list(map(str, command)),
        cwd=PROJECT_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    try:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait(timeout=timeout)
    except Exception:
        process.kill()
        raise

    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)
    return returncode

run_command(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
    enabled=RUN_INSTALL_REQUIREMENTS,
    timeout=1800,
)


## 4. Check sample_700_coarse_manifest.csv

학습 대상 manifest가 라벨별 700개, 총 2,800개인지 확인합니다.

In [ ]:
if not SAMPLE_MANIFEST.exists():
    raise FileNotFoundError(f"sample manifest not found: {SAMPLE_MANIFEST}")

with SAMPLE_MANIFEST.open("r", encoding="utf-8", newline="") as f:
    sample_rows = list(csv.DictReader(f))

print("sample_rows:", len(sample_rows))
print("coarse_label_counts:", dict(Counter(row.get("coarse_label") for row in sample_rows)))
print("split_counts:", dict(Counter(row.get("split") for row in sample_rows)))

print("label_split_counts")
for key, value in sorted(Counter((row.get("coarse_label"), row.get("split")) for row in sample_rows).items()):
    print(key, value)

## 5. Download sampled videos

라벨별 700개, 총 2,800개 영상을 다운로드합니다. 이미 받은 파일은 `exists`로 건너뜁니다.

In [ ]:
run_command(
    [
        sys.executable,
        "etl/download_sampled_media.py",
        "--input", SAMPLE_MANIFEST,
        "--output", DOWNLOAD_MANIFEST,
        "--download-dir", RAW_VIDEO_DIR,
        "--label-column", "coarse_label",
        "--per-label", str(PER_LABEL),
        "--split", "",
    ],
    enabled=RUN_DOWNLOAD,
    timeout=None,
)

## 6. Check download manifest

다운로드 성공/실패 건수와 라벨별 개수를 확인합니다.

In [ ]:
if not DOWNLOAD_MANIFEST.exists():
    raise FileNotFoundError(f"download manifest not found: {DOWNLOAD_MANIFEST}")

with DOWNLOAD_MANIFEST.open("r", encoding="utf-8", newline="") as f:
    download_rows = list(csv.DictReader(f))

print("download_rows:", len(download_rows))
print("coarse_label_counts:", dict(Counter(row.get("coarse_label") for row in download_rows)))
print("download_status_counts:", dict(Counter(row.get("download_status") for row in download_rows)))
print("file_exists_counts:", dict(Counter(row.get("file_exists") for row in download_rows)))

for row in download_rows[:5]:
    print(row.get("coarse_label"), row.get("download_status"), row.get("local_path"))

## 7. Extract training frames

영상 1개당 8프레임을 균등 추출해서 frame-level manifest를 생성합니다.

In [ ]:
run_command(
    [
        sys.executable,
        "etl/extract_training_frames.py",
        "--input", DOWNLOAD_MANIFEST,
        "--output", FRAME_MANIFEST,
        "--frame-dir", FRAME_DIR,
        "--label-column", "coarse_label",
        "--frames-per-video", str(FRAMES_PER_VIDEO),
        "--overwrite",
    ],
    enabled=RUN_EXTRACT_FRAMES,
    timeout=None,
)

## 8. Check frame-level manifest

학습에 들어갈 프레임 수, split, 라벨 분포를 확인합니다.

In [ ]:
if not FRAME_MANIFEST.exists():
    raise FileNotFoundError(f"frame manifest not found: {FRAME_MANIFEST}")

with FRAME_MANIFEST.open("r", encoding="utf-8", newline="") as f:
    frame_rows = list(csv.DictReader(f))

print("frame_rows:", len(frame_rows))
print("coarse_label_counts:", dict(Counter(row.get("coarse_label") for row in frame_rows)))
print("split_counts:", dict(Counter(row.get("split") for row in frame_rows)))
print("extract_status_counts:", dict(Counter(row.get("extract_status") for row in frame_rows)))

missing = [row for row in frame_rows if row.get("frame_exists") != "True"]
print("missing_frame_rows:", len(missing))

for row in frame_rows[:5]:
    print(row.get("split"), row.get("coarse_label"), row.get("frame_path"))

## 9. Train ResNet18 baseline

사전학습 ResNet18 backbone 위에 4-class classifier를 학습합니다. 기본값은 5 epochs, batch size 32입니다.

In [ ]:
def build_train_command(*, epochs, batch_size, learning_rate, freeze_backbone):
    command = [
        sys.executable,
        "ai/vision/train_classifier.py",
        "--manifest", FRAME_MANIFEST,
        "--root-dir", PROJECT_ROOT,
        "--output-dir", MODEL_DIR,
        "--label-column", "coarse_label",
        "--model-name", "resnet18",
        "--image-size", str(IMAGE_SIZE),
        "--epochs", str(epochs),
        "--batch-size", str(batch_size),
        "--learning-rate", str(learning_rate),
        "--seed", str(SEED),
        "--device", DEVICE,
        "--num-workers", str(NUM_WORKERS),
    ]
    if PRETRAINED:
        command.append("--pretrained")
    if freeze_backbone:
        command.append("--freeze-backbone")
    if DETERMINISTIC:
        command.append("--deterministic")
    else:
        command.append("--no-deterministic")
    return command

if RUN_EXPERIMENTS:
    for experiment in EXPERIMENTS:
        print("\n===", experiment["name"], "===")
        train_command = build_train_command(
            epochs=experiment["epochs"],
            batch_size=experiment["batch_size"],
            learning_rate=experiment["learning_rate"],
            freeze_backbone=experiment["freeze_backbone"],
        )
        run_command(train_command, enabled=RUN_TRAIN, timeout=None)
else:
    train_command = build_train_command(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        freeze_backbone=FREEZE_BACKBONE,
    )
    run_command(train_command, enabled=RUN_TRAIN, timeout=None)


## 10. Review latest training result

가장 최근 학습 run의 설정, 라벨 매핑, epoch별 loss/accuracy를 확인합니다.

In [ ]:
runs = sorted([p for p in MODEL_DIR.glob("vision_cls_*") if p.is_dir()])
if not runs:
    raise FileNotFoundError(f"No training runs found under {MODEL_DIR}")

latest_run = runs[-1]
print("latest_run:", latest_run)

config_path = latest_run / "run_config.json"
mapping_path = latest_run / "class_mapping.json"
history_path = latest_run / "training_history.csv"
model_path = latest_run / "model.pt"

print("model_exists:", model_path.exists(), model_path)

if config_path.exists():
    config = json.loads(config_path.read_text(encoding="utf-8"))
    print("run_config")
    for key, value in config.items():
        print(f"- {key}: {value}")

if mapping_path.exists():
    mapping = json.loads(mapping_path.read_text(encoding="utf-8"))
    print()
    print("class_mapping:", mapping)

if history_path.exists():
    with history_path.open("r", encoding="utf-8", newline="") as f:
        history_rows = list(csv.DictReader(f))
    print()
    print("training_history")
    for row in history_rows:
        print(row)


## 11. Final checklist

이 셀의 출력이 모두 `True`면 RunPod 학습 산출물 생성까지 완료된 상태입니다.

In [ ]:
checks = {
    "download_manifest": DOWNLOAD_MANIFEST.exists(),
    "frame_manifest": FRAME_MANIFEST.exists(),
    "model_dir": MODEL_DIR.exists(),
    "latest_model": bool(sorted(MODEL_DIR.glob("vision_cls_*/model.pt"))),
    "latest_history": bool(sorted(MODEL_DIR.glob("vision_cls_*/training_history.csv"))),
}

for key, value in checks.items():
    print(f"{key}: {value}")

print("completed_at:", datetime.now().isoformat(timespec="seconds"))